In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [24]:
batch_size = 128
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=transform
)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

Training samples: 60000
Test samples: 10000


In [25]:
class LocalLayerBlock(nn.Module):
    """
    A 'layer' with K scalar neurons:
      y_i = w_i^T x + b_i, for i in [1..K].
    Each neuron also has a separate classifier (1->10) to measure how well y_i alone predicts the class.
    """
    def __init__(self, in_dim, K=10, num_classes=10):
        super().__init__()
        self.in_dim = in_dim
        self.K = K
        # A single linear transform that outputs K scalars
        self.weight_block = nn.Linear(in_dim, K, bias=True)
        
        # Classification heads: each neuron y_i -> 10 logits
        # We'll do K separate heads, each is a linear(1, 10).
        self.class_heads = nn.ModuleList([nn.Linear(1, num_classes) for _ in range(K)])
    
    def forward(self, x):
        """
        x: (batch_size, in_dim)
        returns:
          Y: shape (batch_size, K), the raw scalar outputs of each neuron
          logits_list: list of length K, each (batch_size, 10)
        """
        # shape (B, K)
        Y = self.weight_block(x)
        
        # For each neuron y_i, pass through its classification head
        logits_list = []
        for i in range(self.K):
            # y_i shape: (B,) => unsqueeze -> (B,1)
            y_i = Y[:, i].unsqueeze(1)
            logits_i = self.class_heads[i](y_i)  # (B, 10)
            logits_list.append(logits_i)
        
        return Y, logits_list
    
    def redundancy_penalty(self):
        """
        Compute sum_{i<j} (w_i dot w_j).
        We'll extract the (in_dim x K) weight matrix from self.weight_block.
        """
        W = self.weight_block.weight  # shape (K, in_dim), 
                                      # but note PyTorch stores shape (out_features, in_features)
        # We want shape (in_dim, K) for easier dot products => just do W.t()
        Wt = W.t()  # shape (in_dim, K)
        
        # sum_{i<j} w_i^T w_j = sum of upper off-diagonal entries of Wt^T Wt
        # But let's just do it directly:
        
        # We can do dot products by Wt^T * Wt => shape (K, K)
        # then sum the off-diagonal
        dot_mat = torch.matmul(Wt.t(), Wt)  # shape (K, K), dot_mat[i,j] = w_i dot w_j
        K_ = dot_mat.shape[0]
        
        # sum of i<j
        off_diag_sum = 0.0
        for i in range(K_):
            for j in range(i+1, K_):
                off_diag_sum += dot_mat[i, j]
        
        return off_diag_sum

In [26]:
def train_local_layer(
    layer_block, 
    train_loader, 
    lr=1e-3, 
    lambda_reg=0.01, 
    epochs=3,
    device='cpu'
):
    """
    Trains the layer with the local cost:
      sum_i CE_i + lambda * sum_{i<j} (w_i dot w_j).
    """
    layer_block.to(device)
    optimizer = optim.Adam(layer_block.parameters(), lr=lr)
    ce_loss_fn = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        layer_block.train()
        total_loss = 0.0
        
        for images, labels in train_loader:
            images = images.view(images.size(0), -1).to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            Y, logits_list = layer_block(images)
            
            # sum cross-entropy for each neuron
            ce_sum = 0.0
            for logits_i in logits_list:
                ce_sum += ce_loss_fn(logits_i, labels)
            
            # redundancy penalty
            red_penalty = layer_block.redundancy_penalty()
            
            loss = ce_sum + lambda_reg * red_penalty
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{epochs}] - Local Layer Loss: {avg_loss:.4f}")
    
    return layer_block

In [27]:
def freeze_module(module):
    """Prevent any gradient updates in this module's parameters."""
    for param in module.parameters():
        param.requires_grad = False

def get_hidden_representation(layer_block, data_loader):
    """
    Pass all data through 'layer_block' to get the K-dimensional hidden representation.
    Return (X_hidden, y_labels) as Tensors on CPU.
    """
    Xs = []
    Ys = []
    layer_block.eval()
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.view(images.size(0), -1).to(device)
            Y, _ = layer_block(images)  # Y shape: (B, K)
            Xs.append(Y.cpu())
            Ys.append(labels)
    return torch.cat(Xs, dim=0), torch.cat(Ys, dim=0)

def build_and_train_2layer_local(train_loader, test_loader, device=device):
    # 1) Train first layer
    layer1 = LocalLayerBlock(in_dim=784, K=16)
    layer1 = train_local_layer(layer1, train_loader, lr=1e-3, lambda_reg=0.01, epochs=3, device=device)
    
    # 2) Freeze first layer
    freeze_module(layer1)
    
    # 3) Extract hidden rep for entire train set
    X_train_hidden, y_train = get_hidden_representation(layer1, train_loader)
    X_test_hidden, y_test = get_hidden_representation(layer1, test_loader)
    
    # Build new DataLoaders from these hidden reps
    train_hidden_dataset = torch.utils.data.TensorDataset(X_train_hidden, y_train)
    test_hidden_dataset = torch.utils.data.TensorDataset(X_test_hidden, y_test)
    
    train_hidden_loader = torch.utils.data.DataLoader(train_hidden_dataset, batch_size=128, shuffle=True)
    test_hidden_loader = torch.utils.data.DataLoader(test_hidden_dataset, batch_size=128, shuffle=False)
    
    # 4) Train second layer (which has K=16 -> each neuron tries to classify c)
    layer2 = LocalLayerBlock(in_dim=16, K=16)
    layer2 = train_local_layer(layer2, train_hidden_loader, lr=1e-3, lambda_reg=0.01, epochs=3, device=device)
    
    return layer1, layer2, train_hidden_loader, test_hidden_loader

In [28]:
def evaluate_second_layer(layer2, data_loader, device='cpu'):
    """
    Evaluate classification by averaging the logits from all K neurons in layer2.
    """
    layer2.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_hid, labels in data_loader:
            X_hid = X_hid.to(device)
            labels = labels.to(device)
            
            Y, logits_list = layer2(X_hid)
            # logits_list is a list of K => shape is (B,10) for each
            stacked_logits = torch.stack(logits_list, dim=0)  # (K, B, 10)
            avg_logits = torch.mean(stacked_logits, dim=0)    # (B, 10)
            
            preds = avg_logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    return 100.0 * correct / total

In [29]:
layer1, layer2, train_hidden_loader, test_hidden_loader = build_and_train_2layer_local(train_loader, test_loader, device=device)

# Evaluate accuracy on test set (using the second layer's ensemble of neurons).
test_acc_local = evaluate_second_layer(layer2, test_hidden_loader, device=device)
print(f"Final 2-layer local learning test accuracy: {test_acc_local:.2f}%")

Epoch [1/3] - Local Layer Loss: 29.4524
Epoch [2/3] - Local Layer Loss: 1.5201
Epoch [3/3] - Local Layer Loss: -51.8815
Epoch [1/3] - Local Layer Loss: 36.5483
Epoch [2/3] - Local Layer Loss: 29.8279
Epoch [3/3] - Local Layer Loss: 28.1628
Final 2-layer local learning test accuracy: 78.28%


In [30]:
class TwoLayerNet(nn.Module):
    def __init__(self, in_dim=784, hidden_dim=64, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [31]:
def train_backprop(model, train_loader, lr=1e-3, epochs=5, device='cpu'):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    ce_loss = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images = images.view(images.size(0), -1).to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            logits = model(images)
            loss = ce_loss(logits, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {running_loss/len(train_loader):.4f}")

def evaluate_accuracy(model, data_loader, device='cpu'):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.view(images.size(0), -1).to(device)
            labels = labels.to(device)
            logits = model(images)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total

In [32]:
model_bp = TwoLayerNet(in_dim=784, hidden_dim=64, num_classes=10)
train_backprop(model_bp, train_loader, lr=1e-3, epochs=5, device=device)

test_acc_bp = evaluate_accuracy(model_bp, test_loader, device=device)
print(f"Standard Backprop 2-layer test accuracy: {test_acc_bp:.2f}%")

Epoch [1/5] - Loss: 0.4830
Epoch [2/5] - Loss: 0.2404
Epoch [3/5] - Loss: 0.1882
Epoch [4/5] - Loss: 0.1549
Epoch [5/5] - Loss: 0.1327
Standard Backprop 2-layer test accuracy: 95.89%
